# EPUB → ZIP с аудиокнигой (<название книги>.m4b + <название книги>-read-along.epub)

Блокнот для запуска из терминала через Colab CLI. Указываете книгу — на выходе один архив
с двумя файлами: аудиокнига `<название книги>.m4b` и текст с подсветкой
`<название книги>-read-along.epub` — имена берутся из названия книги в epub.

## Как запускать

```bash
pip install google-colab-cli          # один раз

colab new -s book --gpu T4                                       # сессия с GPU
colab upload -s book ~/Downloads/kniga.epub /content/kniga.epub  # книга на машину
colab exec -s book -f qwen_tts_book_cli.ipynb                    # этот блокнот
colab download -s book /content/out/kniga.zip ~/Downloads/       # архив к себе
colab stop -s book                                               # освободить машину
```

Скриптовая форма — совсем одной строкой, книга сразу за именем файла:

```bash
colab run --gpu T4 --keep -s book colab/cli/qwen_tts_book.py "kniga.epub"
```

Если Google Drive подключён (`colab drivemount -s book`), архив дополнительно копируется в
`MyDrive/audiobooks/` — тогда `colab download` не нужен.

## Что делает

1. ставит зависимости и берёт веса модели синтеза (один раз на машину);
2. строит манифест из EPUB, расставляет ударения, собирает библиотеку голоса чтеца;
3. синтезирует речь, выравнивает громкость и мягко обрабатывает её (свист на «с/ш/ч» тише),
   подмешивает музыку;
4. пакует `<название книги>.m4b` + `<название книги>-read-along.epub` в один zip и печатает путь, размер и sha256.

Каталог проекта у каждой книги свой (`/content/out/<имя книги>`), поэтому кэш сегментов разных
книг не перемешивается, а повторный запуск продолжает с места остановки.

## Параметры

`BOOK` можно не трогать: имя подхватится из аргумента командной строки, из переменной окружения
`BOOK`, либо найдётся единственный epub в `/content` или `MyDrive`.

In [ ]:
import os, pathlib, sys

BOOK = "idiot.epub"          # имя файла, путь или ссылка; "" → искать автоматически
SECONDS = None               # None → вся книга; 300 → только пять минут (быстрая проверка)
HIGHLIGHT = "sentence"       # "sentence" — подсветка по фразам; "fragment" — по абзацам (быстрее)
MODEL_DIR = "/content/models/Qwen3-TTS-12Hz-1.7B-Ru-Stress-CF-source"
LOCAL = sys.platform == "darwin"     # на macOS прогон идёт на Metal, без установки пакетов


def pick_book():
    given = [os.environ.get("BOOK", "").strip(), BOOK.strip()]
    given += [a for a in sys.argv[1:] if a.lower().endswith(".epub")]
    for candidate in given:
        if candidate:
            print("книга из параметров:", candidate)
            return candidate
    roots = [pathlib.Path("/content"), pathlib.Path("/content/drive/MyDrive")]
    found = [p for root in roots if root.is_dir() for p in root.glob("*.epub")]
    if len(found) == 1:
        print("книга найдена автоматически:", found[0])
        return str(found[0])
    print("не задано (BOOK пуст) и в /content не один epub:", [p.name for p in found])
    return ""


BOOK_PATH = pick_book()
print("книга        :", BOOK_PATH or "НЕ ЗАДАНА")
print("фрагмент     :", f"{SECONDS} с" if SECONDS else "вся книга")
print("подсветка    :", HIGHLIGHT)
print("веса модели  :", MODEL_DIR, "| локальный режим:", LOCAL)

In [ ]:
import os, shutil, subprocess, sys

print("Python:", sys.version.split()[0], "| ffmpeg:", shutil.which("ffmpeg") or "нет")
gpu = ""
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=20).stdout.strip()
except Exception as exc:
    print("nvidia-smi:", exc)
if not gpu:
    try:
        import torch
        if torch.cuda.is_available():
            gpu = torch.cuda.get_device_name(0)
    except Exception as exc:
        print("torch:", exc)
if not gpu:
    try:
        import mlx.core as mx
        if "gpu" in str(mx.default_device()).lower():
            gpu = f"MLX {mx.device_info().get('device_name')}"
    except Exception as exc:
        print("mlx:", exc)
print("GPU   :", gpu or "НЕ НАЙДЕН")
assert gpu, ("GPU не найден. Запустите сессию с ускорителем: colab new -s book --gpu T4, "
             "либо colab run --gpu T4 --keep ...")
#  Кэши MLX/CUDA считаются в записях, а не в мегабайтах: по умолчанию свёртки 128, граф 400,
#  внимание 256/64, FFT 128. Когда записей не хватает, MLX печатает «Cache thrashing is happening,
#  please set the environment variable <имя> to a larger value than <N>» и синтез падает на середине
#  книги. Ставим потолки с запасом (это лимит на число записей, а не резервация памяти).
CACHE_ENTRIES = {"MLX_CUDA_CONV_CACHE_SIZE": 12288, "MLX_CUDA_GRAPH_CACHE_SIZE": 8192,
                 "MLX_CUDA_SDPA_CACHE_SIZE": 4096, "MLX_CUDA_SDPA_BACKWARD_CACHE_SIZE": 1024,
                 "MLX_CUDA_FFT_CACHE_SIZE": 1024}
for _name, _value in CACHE_ENTRIES.items():
    os.environ.setdefault(_name, str(_value))
print("кэши MLX/CUDA:", ", ".join(f"{name.split('_')[2].lower()}={value}"
                                   for name, value in CACHE_ENTRIES.items()))

In [ ]:
import pathlib, subprocess

REPO = pathlib.Path("/content/qwen-epub-audiobook")
if (REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "-q"], check=False)
    print("репозиторий обновлён:", REPO)
else:
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/davnozdu/qwen-epub-audiobook.git", str(REPO)], check=True)
    print("репозиторий склонирован:", REPO)
job = REPO / "colab" / "cli" / "qwen_tts_book.py"
assert job.is_file(), f"нет {job}"
print("задание:", job)

In [ ]:
import pathlib, subprocess, sys

assert BOOK_PATH, "BOOK не задан: впишите имя файла в ячейку параметров или положите epub в /content"
stem = pathlib.Path(BOOK_PATH.split("?")[0]).stem
base = pathlib.Path("/content/out") if pathlib.Path("/content").is_dir() else pathlib.Path.cwd() / "out"
command = [sys.executable, str(job), BOOK_PATH, "--highlight", HIGHLIGHT, "--model-dir", MODEL_DIR,
           "--out", str(base / stem), "--zip", str(base / f"{stem}.zip")]
if SECONDS:
    command += ["--seconds", str(SECONDS)]
if LOCAL:
    command += ["--local"]
print("+", " ".join(command), flush=True)

process = subprocess.Popen(command, cwd=str(REPO), text=True, bufsize=1,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for line in process.stdout:
    print(line.rstrip(), flush=True)
process.wait()
assert process.returncode == 0, f"конвейер упал с кодом {process.returncode}"

In [ ]:
import hashlib, pathlib, zipfile

archive = pathlib.Path(base / f"{stem}.zip")
assert archive.is_file(), f"архива нет: {archive}"
print("архив :", archive)
print("размер:", f"{archive.stat().st_size / 1e6:.1f} МБ")
print("sha256:", hashlib.sha256(archive.read_bytes()).hexdigest())
with zipfile.ZipFile(archive) as zf:
    for info in zf.infolist():
        print(f"  {info.filename:24} {info.file_size / 1e6:7.1f} МБ")

drive = pathlib.Path("/content/drive/MyDrive/audiobooks") / archive.name
if drive.is_file():
    print("\nна Google Drive:", drive)
else:
    print("\nскачать на свой диск:\n  colab download -s <сессия>", archive, f"~/Downloads/{archive.name}")

## Если что-то не так

- `GPU не найден` — сессия без ускорителя: `colab new -s book --gpu T4`.
- `книга не найдена` — файл не на машине: `colab upload -s book ~/Downloads/kniga.epub /content/kniga.epub`.
- `Cache thrashing` — уже обработано: потолок кэша свёрток 12288 МБ и автоповтор с 16384.
- Повторный запуск продолжает с места остановки по кэшу сегментов; чтобы пересобрать с нуля,
  удалите каталог `/content/out/<имя книги>` (`colab rm -s book -r /content/out/kniga`).